# Single Name Stock Data

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
note_path = os.getcwd()
repo_path = os.path.abspath(os.path.join(note_path,".."))
data_path = os.path.join(repo_path, "data")
fact_path = os.path.join(data_path, "FactorAnalysis")

In [23]:
group_renamer_ = {
    'em_carry'       : "EM Carry", 
    'g10_carry'      : "G10 Carry", 
    'illiquid_future': "Ill Fut", 
    'liquid_future'  : "Liq Fut",
    "lag_signal"     : "Lagged",
    "signal"         : "No Lag",
    "coeff_val"      : r"$\beta$",
    "rsquared"       : r"$R^2$",
    "full_sample"    : "Full-Sample", 
    'in_sample'      : "In-Sample", 
    'out_sample'     : "Out-Sample",
    "train_test"     : "Train/Test",
    "lag_vol_rtn"    : "Lagged",	
    "vol_rtn"        : "Perfect"}

renamer = {**group_renamer_}
FLOAT_FORMAT = lambda x: f"{x:.2f}"

In [4]:
single_path = os.path.join(data_path, "Signals", "SingleNameStockSpreadTrend.parquet")
df_single   = (pd
    .read_parquet(path = single_path, engine = "pyarrow"))

In [5]:
df_ticker_count = (df_single
    [["stock_ticker", "etf_ticker"]]
    .groupby(["etf_ticker", "stock_ticker"])
    .head(1)
    .groupby("etf_ticker")
    .agg("count")
    .sort_values("stock_ticker"))

In [17]:
path      = os.path.join(data_path, "FXTickerGuide.xlsx")
df_ticker = (pd
    .read_excel(io = path)
    .loc[lambda x: x.Active == True]
    [["ticker", "etf_ticker", "group"]]
    .assign(etf_ticker = lambda x: x.etf_ticker.str.split(" ").str[0])
    .dropna()
    .rename(columns = {"ticker": "fx_ticker"}))

In [20]:
display(df_ticker
    .drop(columns = ["group"])
    .merge(right = df_ticker_count, how = "inner", on = ["etf_ticker"])
    .sort_values("stock_ticker")
    .assign(fx_ticker = lambda x: "-" + x.fx_ticker)
    .groupby(["etf_ticker", "stock_ticker"])
    .agg("sum")
    .assign(fx_ticker = lambda x: x.fx_ticker.str[1:])
    .reset_index()
    .sort_values("stock_ticker"))

,etf_ticker,stock_ticker,fx_ticker
16,EWT,7,USDTWDCR Curncy
15,EWS,11,USDSGDCR Curncy
8,EPU,11,USDPENCR Curncy
0,COLO,16,USDCOPCR Curncy
12,EWH,18,USDHKDCR Curncy
21,EZA,22,USDZARCR Curncy
1,ECH,22,USDCLPCR Curncy
6,ENZL,24,USDNZDCR Curncy-NV1 Curncy
7,EPOL,25,USDPLNCR Curncy-PP1 Curncy
18,EWW,26,PE1 Curncy-USDMXNCR Curncy


In [22]:
path = r"A:\GitHub\FXEquitySpillover\data\FactorAnalysis\SingleStockETFModelComparison.parquet"
display(pd
    .read_parquet(path = path, engine = "pyarrow")
    .drop(columns = ["pvalue", "tvalue", "etf_ticker"])
    .melt(id_vars = ["name", "fx_ticker", "lag"])
    .pivot(index = ["fx_ticker", "lag", "variable"], columns = "name", values = "value")
    .assign(spread = lambda x: x.stock - x.etf)
    .reset_index()
    .merge(right = df_ticker, how = "inner", on = ["fx_ticker"])
    [["lag", "variable", "group", "spread"]]
    .groupby(["lag", "variable", "group"])
    .agg("mean")
    .reset_index()
    .replace(renamer)
    .rename(columns = {
        "group"   : "Group",
        "variable": "",
        "lag"     : "Lag"})
    .pivot(index = ["Group"], columns = ["Lag", ""], values = "spread"))

Lag          Lagged              No Lag          
            $\beta$     $R^2$   $\beta$     $R^2$
Group                                            
EM Carry   0.000378  0.014315  0.002632 -0.084769
G10 Carry -0.000065  0.035083  0.002982 -0.039104
Ill Fut   -0.000122  0.042125 -0.002699 -0.103750
Liq Fut   -0.000098  0.031470 -0.003065 -0.046648

In [9]:
path = r"A:\GitHub\FXEquitySpillover\data\FXTickerGuide.xlsx"
df_ticker = (pd
    .read_excel(io = path, sheet_name = "TickerGuide")
    [["ticker", "group"]]
    .rename(columns = {"ticker": "fx_ticker"}))

In [10]:
path = r"A:\GitHub\FXEquitySpillover\data\Backtests\SingleNameOLSForecasted.parquet"
df_combined = (pd
    .read_parquet(path = path, engine = "pyarrow")
    .merge(right = df_ticker, how = "inner", on = ["fx_ticker"]))

In [11]:
df_port = (df_combined
    [["fx_ticker", "sample_group", "regression", "group", "date", "vol_rtn", "lag_vol_rtn"]]
    .melt(id_vars = ["date", "fx_ticker", "sample_group", "regression", "group"])
    .dropna()
    .drop(columns = ["fx_ticker"])
    .groupby(["date", "sample_group", "regression", "group", "variable"])
    .agg("mean"))

In [12]:
df_sharpe = (df_port
    .reset_index()
    .drop(columns = ["date"])
    .groupby(["sample_group", "regression", "group", "variable"])
    .agg(lambda x: x.mean() / x.std() * np.sqrt(252))
    .reset_index())

In [27]:
display(df_sharpe
    .loc[lambda x: x.variable == "vol_rtn"]
    .loc[lambda x: x.regression != "expanding"]
    .replace(renamer)
    .rename(columns = {
        "group"       : "Group",
        "regression"  : "Model",
        "sample_group": "Sample Group"})
    .pivot(index = "Group", columns = ["Model", "Sample Group"], values = "value"))

Model        Full-Sample Train/Test           
Sample Group Full-Sample  In-Sample Out-Sample
Group                                         
EM Carry        2.988994   3.043586   0.992377
G10 Carry       1.853729   2.013241  -0.817259
Ill Fut         3.384526   3.668195   0.188843
Liq Fut         2.555154   2.797608   0.164892

In [14]:
df_tmp_port = (df_port
    .reset_index()
    .drop(columns = ["sample_group"])
    .groupby(["date", "regression", "group", "variable"])
    .agg("mean")
    .reset_index())

In [31]:
display(df_tmp_port
    .drop(columns = ["date"])
    .groupby(["regression", "group", "variable"])
    .agg(lambda x: x.mean() / x.std() * np.sqrt(252))
    .reset_index()
    .loc[lambda x: x.regression != "expanding"]
    .replace(renamer)
    .pivot(index = "group", columns = ["regression", "variable"], values = "value"))

regression Full-Sample           Train/Test          
variable        Lagged   Perfect     Lagged   Perfect
group                                                
EM Carry      2.647845  2.988994   2.380468  2.625470
G10 Carry     1.787527  1.853729   1.470376  1.562371
Ill Fut       3.279232  3.384526   2.707343  2.773282
Liq Fut       2.481189  2.555154   2.200058  2.291719

In [32]:
print(os.getcwd())

A:\GitHub\FXEquitySpillover\notebook
